In [1]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

import pandas as pd

from preprocessing import prepare_model_data
from modeling import (
    train_and_compare_models,
    retrain_best_model,
    test_model,
    summarize_results,
    build_prediction_comparison,
    save_model
)

In [2]:
# prepare data from the raw model input file
X_train, y_train, X_val, y_val, X_test, y_test = prepare_model_data(
    "../data/raw/model_input_daily_2024_2025.csv"
)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(510, 29) (110, 29) (110, 29)
(510,) (110,) (110,)


In [3]:
# experiment 01: compare baseline candidate models on validation data
results_df, fitted_pipelines = train_and_compare_models(X_train, y_train, X_val, y_val)

results_df

,MAE,RMSE,R2,Model
0,564.934474,731.584943,0.923316,LinearRegression
1,640.426602,806.987540,0.906694,GradientBoosting
2,645.864222,844.789806,0.897748,Ridge
3,778.698579,941.074419,0.873111,RandomForest


In [4]:
# choose the best model based on lowest validation RMSE
best_model_name = results_df.loc[0, "Model"]
best_pipeline = fitted_pipelines[best_model_name]

print("Best validation model:", best_model_name)

Best validation model: LinearRegression


In [5]:
# grab validation metrics for the selected model
val_metrics = (
    results_df.loc[results_df["Model"] == best_model_name, ["MAE", "RMSE", "R2"]]
    .iloc[0]
    .to_dict()
)

val_metrics

{'MAE': 564.9344742534054, 'RMSE': 731.5849431340137, 'R2': 0.923315910693593}

In [6]:
# retrain the selected model on train + validation data
best_pipeline = retrain_best_model(best_pipeline, X_train, y_train, X_val, y_val)

In [7]:
# evaluate on the held-out test set
test_preds, test_metrics = test_model(best_pipeline, X_test, y_test)

test_metrics

{'MAE': 596.907327366643,
 'RMSE': np.float64(765.3470341041038),
 'R2': 0.8874344140489053}

In [8]:
# summarize validation and test performance
summary_df = summarize_results(best_model_name, val_metrics, test_metrics)

summary_df

,Selected_Model,Validation_MAE,Validation_RMSE,Validation_R2,Test_MAE,Test_RMSE,Test_R2
0,LinearRegression,564.934474,731.584943,0.923316,596.907327,765.347034,0.887434


In [9]:
# build a prediction comparison table
pred_comparison = build_prediction_comparison(y_test, test_preds)

pred_comparison.head()

,actual,predicted,error,abs_error
0,24568.850694,24516.729568,52.121126,52.121126
1,24145.447917,24441.333443,-295.885526,295.885526
2,28465.854167,27838.624769,627.229398,627.229398
3,30761.111111,30176.352869,584.758242,584.758242
4,31047.944444,31087.872746,-39.928301,39.928301


In [10]:
# save model
model_path = "../data/artifacts/ca_electricity_demand_lr_v1.joblib"
save_model(best_pipeline, model_path)

## Modeling Summary & Key Takeaways

- I tested several baseline models on the validation set, including linear regression, ridge, random forest, and gradient boosting. Linear regression ended up performing the best overall.

- Linear regression achieved:
  - Validation R² ≈ 0.92  
  - Test R² ≈ 0.88  
  - RMSE ≈ 765 MW on the test set  

- The drop from validation to test performance is expected and suggests the model generalizes reasonably well without severe overfitting.

- More complex models like random forest and gradient boosting did not outperform linear regression. This suggests that the relationship between features (especially temperature and time-based features) and load is largely linear or well-captured by the engineered features.

- Feature engineering played a major role in performance:
  - Aggregated multi-city weather into statewide features (temperature, humidity, precipitation, wind)
  - Created temperature-driven features:
    - cooling degree and heating degree
    - squared terms to capture nonlinearity
    - hot / very hot indicators
  - Added calendar-based features:
    - seasonal flags (summer, winter, shoulder)
    - weekend indicator
  - Used cyclical encodings for time:
    - month, day of week, and day of year (sin/cos transforms)
  - Built interaction features:
    - temperature × season (e.g. summer cooling)
    - temperature × weekend
    - temperature × humidity
  - Included a lag-1 load feature to capture short-term persistence in demand

- I also removed same-day load summary columns to avoid leakage, so the model relies on weather, calendar structure, and prior load rather than information from the target day itself.

- After selecting the best model, I retrained it on the combined train + validation data to maximize the training signal before evaluating on the held-out test set.

- Final model was saved as:
  - `ca_electricity_demand_lr_v1.joblib`

---

**Takeaway:**  
Electricity demand in California is driven primarily by temperature, seasonality, and short-term patterns. With the right feature engineering, a relatively simple linear model is able to capture these relationships effectively and generalize well to unseen data.